In [5]:
import torch
import pandas as pd
import numpy as np
import random
import ast
from transformers import BertTokenizer, BertForMaskedLM
from tqdm.auto import tqdm
import gc

# --- CẤU HÌNH DANH SÁCH 3 MODEL ---
MODEL_PATHS = [
    "/kaggle/input/model-final/WWM_Update/WWM_Update", 
    "/kaggle/input/model-final/CLM_update/CLM_update",
    "/kaggle/input/model-final/Hybrid_Update/Hybrid_Update"
]

MODEL_NAMES = ["WWM", "CLM", "Hybrid WWM + CLM"]

FILE_CORPUS = "/kaggle/input/final-prj-nlp/test_prose.xlsx"
FILE_SIMILAR = "/kaggle/input/final-prj-nlp/SinoNom_similar_Dic.xlsx"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Device: {DEVICE}")

def load_data_resources():
    print("Dang load Corpus & Dictionary...")
    df = pd.read_excel(FILE_CORPUS)
    raw_texts = df['Text'].dropna().astype(str).tolist()
    texts = [t for t in raw_texts if len(t) > 7]
    try:
        df_sim = pd.read_excel(FILE_SIMILAR)
        sim_dict = {}
        for _, row in df_sim.iterrows():
            k = str(row.iloc[0]).strip()
            try: vals = ast.literal_eval(str(row.iloc[1]))
            except: vals = [x.strip() for x in str(row.iloc[1]).replace('[','').replace(']','').replace("'","").split(',')]
            sim_dict[k] = vals
    except: sim_dict = {}
    print(f"Corpus: {len(texts)} cau | Similar Dict: {len(sim_dict)} muc")
    return texts, sim_dict

GLOBAL_TEXTS, GLOBAL_SIM_DICT = load_data_resources()
# Lấy mẫu test cố định 2000 câu cho Prose
TEST_TEXTS = random.sample(GLOBAL_TEXTS, min(2000, len(GLOBAL_TEXTS)))

def generate_hard_samples(texts, tokenizer, sim_dict, length_mode='1', task_type='Replacement'):
    samples = []
    for text in texts:
        tokens = tokenizer.tokenize(text)
        seq_len = len(tokens)
        if length_mode == '1': k = 1
        elif length_mode == '2': k = 2
        else: k = random.randint(3, 5)
        if seq_len <= k + 2: continue
        
        candidates = []
        if task_type == 'Replacement' and sim_dict:
            for i in range(1, seq_len - k - 1):
                if any(t in sim_dict for t in tokens[i : i+k]): candidates.append(i)
        
        start_idx = random.choice(candidates) if candidates else random.randint(1, seq_len - k - 1)
        end_idx = start_idx + k
        target_tokens = tokens[start_idx:end_idx]
        masked_tokens = tokens.copy()
        for i in range(start_idx, end_idx): masked_tokens[i] = "[MASK]"
        samples.append({
            'input_ids': tokenizer.convert_tokens_to_ids(masked_tokens),
            'target_ids': tokenizer.convert_tokens_to_ids(target_tokens),
            'mask_indices': list(range(start_idx, end_idx))
        })
    # Tăng số lượng mẫu test cho mỗi độ dài lên 500 mẫu
    return samples[:500]

def evaluate_batch(model, tokenizer, samples):
    if not samples: return 0.0, 0.0
    p1, p10, total = 0, 0, 0
    batch_size = 16
    for i in range(0, len(samples), batch_size):
        batch = samples[i:i+batch_size]
        max_len = max([len(b['input_ids']) for b in batch])
        input_tensor = torch.full((len(batch), max_len), tokenizer.pad_token_id, dtype=torch.long).to(DEVICE)
        attn_mask = torch.zeros((len(batch), max_len), dtype=torch.long).to(DEVICE)
        for j, item in enumerate(batch):
            ln = len(item['input_ids'])
            input_tensor[j, :ln] = torch.tensor(item['input_ids'])
            attn_mask[j, :ln] = 1
        with torch.no_grad():
            logits = model(input_tensor, attention_mask=attn_mask).logits
        for j, item in enumerate(batch):
            mask_indices, target_ids = item['mask_indices'], item['target_ids']
            is_p1, is_p10 = True, True
            for idx_in_seq, target_id in zip(mask_indices, target_ids):
                probs = logits[j, idx_in_seq, :]
                top10 = torch.topk(probs, 10).indices.tolist()
                if top10[0] != target_id: is_p1 = False
                if target_id not in top10: is_p10 = False
            if is_p1: p1 += 1
            if is_p10: p10 += 1
            total += 1
    return (p1/total * 100), (p10/total * 100)

def run_prose_task(model, tokenizer):
    tasks = ['Insertion', 'Replacement']
    lengths = {'1': 'L1', '2': 'L2', '>3': 'L3'}
    results = {}
    for task in tasks:
        p1s, p10s = [], []
        for l_key, l_label in lengths.items():
            samples = generate_hard_samples(TEST_TEXTS, tokenizer, GLOBAL_SIM_DICT, l_key, task)
            a1, a10 = evaluate_batch(model, tokenizer, samples)
            results[f"{task}_{l_label}_P1"] = a1
            results[f"{task}_{l_label}_P10"] = a10
            p1s.append(a1); p10s.append(a10)
        results[f"{task}_Avg_P1"] = sum(p1s)/len(p1s)
        results[f"{task}_Avg_P10"] = sum(p10s)/len(p10s)
    return results

Device: cuda
Dang load Corpus & Dictionary...
Corpus: 3067 cau | Similar Dict: 26044 muc


In [6]:
from torch.nn.functional import cosine_similarity
FILE_TEST_POETRY = "/kaggle/input/final-prj-nlp/test_poetry.xlsx"
FILE_DICT = "/kaggle/input/final-prj-nlp/QuocNgu_SinoNom_Dic.xlsx"

def load_poetry_resources():
    print("Dang load Poetry Data...")
    try:
        df = pd.read_excel(FILE_TEST_POETRY)
        texts = df['Text'].dropna().astype(str).tolist()
        pairs = [(texts[i].strip(), texts[i+1].strip()) for i in range(0, len(texts)-1, 2) if len(texts[i]) > 3]
        return pairs, texts
    except: return [], []

GLOBAL_POETRY_PAIRS, GLOBAL_ALL_POETRY = load_poetry_resources()
# Tăng số lượng cặp thơ dùng để test lên 1000 cặp
TEST_POETRY_PAIRS = GLOBAL_POETRY_PAIRS[:1000]

def load_tonal_dict():
    print("Dang load Tonal Dictionary...")
    n2v = {}
    try:
        df = pd.read_excel(FILE_DICT)
        for _, row in df.iterrows():
            v, n_raw = str(row.iloc[0]).strip().lower(), str(row.iloc[1]).strip()
            try: n_list = ast.literal_eval(n_raw)
            except: n_list = [x.strip() for x in n_raw.split(',')] if ',' in n_raw else [n_raw]
            for n in n_list:
                if n not in n2v: n2v[n] = []
                if v not in n2v[n]: n2v[n].append(v)
    except: pass
    return n2v

GLOBAL_NOM2VIET = load_tonal_dict()

def get_emb(model, tokenizer, text):
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=64).to(DEVICE)
    with torch.no_grad():
        out = model.bert(**inputs)
        mask = inputs['attention_mask'].unsqueeze(-1)
        return torch.sum(out.last_hidden_state * mask, dim=1) / torch.clamp(mask.sum(dim=1), min=1e-9)

def run_poetry_matching(model, tokenizer):
    correct, total = 0, 0
    # Chạy test Matching trên 500 mẫu
    for s_a, t_b in TEST_POETRY_PAIRS[:500]:
        e_a = get_emb(model, tokenizer, s_a)
        cands = [t_b] + random.sample([t for t in GLOBAL_ALL_POETRY if t != t_b], 3)
        random.shuffle(cands)
        scores = [cosine_similarity(e_a, get_emb(model, tokenizer, c)).item() for c in cands]
        if cands[np.argmax(scores)] == t_b: correct += 1
        total += 1
    return (correct/total * 100) if total > 0 else 0

def run_tonal_check(model, tokenizer):
    correct, total = 0, 0
    trac = "áắấéếíóốớúứýảẳẩẻểỉỏổởủửỷãẵẫẽễĩõỗỡũữỹạặậẹệịọộợụựỵ"
    # Chạy test Luật Bằng Trắc trên 1000 mẫu
    for s_a, s_b in TEST_POETRY_PAIRS[:1000]:
        inputs = tokenizer(s_a, s_b, return_tensors='pt', truncation=True, max_length=128).to(DEVICE)
        ids = inputs['input_ids'][0]
        seps = (ids == tokenizer.sep_token_id).nonzero(as_tuple=True)[0]
        if len(seps) < 2: continue
        target_idx = list(range(seps[0].item() + 1, seps[1].item()))
        if not target_idx: continue
        m_idx = random.choice(target_idx)
        orig_char = tokenizer.decode([ids[m_idx].item()]).strip()
        ids[m_idx] = tokenizer.mask_token_id
        with torch.no_grad():
            pred_id = torch.argmax(model(input_ids=ids.unsqueeze(0)).logits[0, m_idx, :]).item()
            pred_char = tokenizer.decode([pred_id]).strip()
        o_prons, p_prons = GLOBAL_NOM2VIET.get(orig_char, []), GLOBAL_NOM2VIET.get(pred_char, [])
        if o_prons and p_prons:
            o_tone = "TRAC" if any(c in trac for c in o_prons[0]) else "BANG"
            if any(("TRAC" if any(c in trac for c in p) else "BANG") == o_tone for p in p_prons): correct += 1
            total += 1
    return (correct/total * 100) if total > 0 else 0

Dang load Poetry Data...
Dang load Tonal Dictionary...


In [7]:
final_comparison = []
print("BAT DAU DANH GIA CAC MODEL...")

for idx, path in enumerate(MODEL_PATHS):
    name = MODEL_NAMES[idx]
    print(f"\n[{idx+1}/3] Loading: {name}")
    try:
        tk = BertTokenizer.from_pretrained(path)
        md = BertForMaskedLM.from_pretrained(path).to(DEVICE).eval()
        
        prose = run_prose_task(md, tk)
        p_match = run_poetry_matching(md, tk)
        p_tonal = run_tonal_check(md, tk)
        
        res = {"Model": name}
        res.update(prose)
        res.update({"Poetry_Matching": p_match, "Poetry_Tonal": p_tonal})
        final_comparison.append(res)
        
        del md, tk; torch.cuda.empty_cache(); gc.collect()
        print(f"  -> Xong {name}.")
    except Exception as e:
        print(f"  -> Loi {name}: {e}")

print("\nDA HOAN THANH.")

BAT DAU DANH GIA CAC MODEL...

[1/3] Loading: WWM
  -> Xong WWM.

[2/3] Loading: CLM
  -> Xong CLM.

[3/3] Loading: Hybrid WWM + CLM
  -> Xong Hybrid WWM + CLM.

DA HOAN THANH.


In [8]:
print("\nBANG TONG HOP KET QUA SO SANH 3 MODEL:")
if final_comparison:
    df = pd.DataFrame(final_comparison)
    # Xoay dọc bảng: Index là tên các cột cũ (thông số), Cột là tên Model
    df_t = df.set_index("Model").T
    df_t = df_t.reset_index()
    df_t.columns.name = None
    df_t = df_t.rename(columns={"index": "Tên thông số"})
    
    # Làm tròn và hiển thị
    print(df_t.round(2).to_markdown(index=False))
else:
    print("Khong co du lieu.")


BANG TONG HOP KET QUA SO SANH 3 MODEL:
| Tên thông số        |   WWM |   CLM |   Hybrid WWM + CLM |
|:--------------------|------:|------:|-------------------:|
| Insertion_L1_P1     | 43.8  | 42.6  |              45.6  |
| Insertion_L1_P10    | 72.8  | 70.4  |              72.6  |
| Insertion_L2_P1     | 12.8  | 18.4  |              13.4  |
| Insertion_L2_P10    | 38    | 44.8  |              36.6  |
| Insertion_L3_P1     |  1.6  |  2    |               2.8  |
| Insertion_L3_P10    | 11.8  | 11.2  |              12.8  |
| Insertion_Avg_P1    | 19.4  | 21    |              20.6  |
| Insertion_Avg_P10   | 40.87 | 42.13 |              40.67 |
| Replacement_L1_P1   | 40.6  | 42.6  |              42.8  |
| Replacement_L1_P10  | 68.8  | 70.6  |              70    |
| Replacement_L2_P1   | 15.8  | 12.4  |              14.4  |
| Replacement_L2_P10  | 40.6  | 39    |              40.2  |
| Replacement_L3_P1   |  2.4  |  1.2  |               3    |
| Replacement_L3_P10  | 10.2  | 12    |      